## Overview

1. Clustering               pair shortlisting, Dinov2
2. Local matching           keypoints + matcher ALIKED/SuperPoint + LightGlue, or MASt3R
3. Geometric verification   RANSAC 
4. Reconstruction           COLMAP/pycolmap structure-from-motion (SfM) → camera poses
                            each separate reconstruction = one scene cluster; unregistered = outlier

## Pair shortlisting

Pair sortlisting via embedding using Dinov2
https://github.com/facebookresearch/dinov2

embed Images and group them into sets of similar images,
return a list of candidate pairs to match

In [5]:
import torch
import torch.nn.functional as F
from PIL import Image
import torchvision.transforms as T

dinov2 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
device = torch.device('mps')

Using cache found in /Users/lucbaumeler/.cache/torch/hub/facebookresearch_dinov2_main


In [ ]:
dinov2.eval().to(device)

transform = T.Compose([T.Resize((448, 448)), T.ToTensor(), T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))])



DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
    (norm): Identity()
  )
  (blocks): ModuleList(
    (0-11): 12 x NestedTensorBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True, bias=True)
      (attn): MemEffAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (drop_path1): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True, bias=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
      (ls2): LayerScale()
      (drop_path2): Identity()
    )
  )
  (norm): LayerNorm((384,), eps=1